Gender Prediction for Targeted Advertising

In [ ]:
# Import Libraries
!pip install lightgbm py7zr user-agents --quiet
import numpy as np
import pandas as pd
from user_agents import parse
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import lightgbm as lgb
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load datasets with semicolon separator
train_labels = pd.read_csv('/content/drive/train_labels.csv', sep=';')
train_data = pd.read_csv('/content/drive/train.csv', sep=';')
test_data = pd.read_csv('/content/drive/test.csv', sep=';')
referer_vectors = pd.read_csv('/content/drive/referer_vectors.csv', sep=';')
geo_info = pd.read_csv('/content/drive/geo_info.csv', sep=';')
test_users = pd.read_csv('/content/drive/test_users.csv', sep=';')

# Verify column names after proper loading
print("Train data columns:", train_data.columns.tolist())
print("Train data shape:", train_data.shape)
print("Referer vectors columns:", referer_vectors.columns.tolist())
print("Geo info columns:", geo_info.columns.tolist())
print("Train labels columns:", train_labels.columns.tolist())

In [ ]:
# Data Preprocessing
print("\nStarting data preprocessing...")

# Merge referer vectors (component features)
print("Merging referer vectors...")
train_data = train_data.merge(referer_vectors, on='referer', how='left')
test_data = test_data.merge(referer_vectors, on='referer', how='left')

# Merge geo information
print("Merging geo information...")
train_data = train_data.merge(geo_info, on='geo_id', how='left')
test_data = test_data.merge(geo_info, on='geo_id', how='left')

print("After merging - Train data shape:", train_data.shape)
print("After merging - Test data shape:", test_data.shape)

# Parse user agents
def parse_user_agent(ua_string):
    try:
        if pd.isna(ua_string):
            return pd.Series({
                'browser': 'Unknown',
                'os': 'Unknown',
                'is_mobile': 0,
                'is_tablet': 0
            })
        ua = parse(str(ua_string))
        return pd.Series({
            'browser': ua.browser.family,
            'os': ua.os.family,
            'is_mobile': 1 if ua.is_mobile else 0,
            'is_tablet': 1 if ua.is_tablet else 0
        })
    except:
        return pd.Series({
            'browser': 'Unknown',
            'os': 'Unknown',
            'is_mobile': 0,
            'is_tablet': 0
        })

print("Parsing user agents...")
for df in [train_data, test_data]:
    ua_features = df['user_agent'].apply(parse_user_agent)
    df[['browser', 'os', 'is_mobile', 'is_tablet']] = ua_features

# Extract time features from timestamp
print("Extracting time features...")
for df in [train_data, test_data]:
    df['request_ts'] = pd.to_datetime(df['request_ts'], unit='s')
    df['hour'] = df['request_ts'].dt.hour
    df['weekday'] = df['request_ts'].dt.weekday
    df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

# Drop unnecessary columns
cols_to_drop = ['request_ts', 'user_agent', 'referer']
train_data.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')
test_data.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')

print("Data preprocessing completed!")
print("Final train data columns:", train_data.columns.tolist())

In [ ]:
# Feature Engineering
print("\nStarting feature engineering...")

def aggregate_features(df):
    """Aggregate user-level features"""
    # Group by user_id and aggregate features
    grouped = df.groupby('user_id')

    # Initialize aggregated dataframe
    agg_df = pd.DataFrame({'user_id': df['user_id'].unique()})

    # Aggregate components (mean)
    components = [f'component{i}' for i in range(10)]
    for i, comp in enumerate(components):
        if comp in df.columns:
            comp_mean = grouped[comp].mean().reset_index()
            comp_mean.columns = ['user_id', f'comp_mean_{i}']
            agg_df = agg_df.merge(comp_mean, on='user_id', how='left')

    # Add most frequent features (mode)
    categorical_cols = ['country_id', 'region_id', 'timezone_id', 'browser', 'os']
    for col in categorical_cols:
        if col in df.columns:
            mode_values = grouped[col].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown').reset_index()
            mode_values.columns = ['user_id', f'mode_{col}']
            agg_df = agg_df.merge(mode_values, on='user_id', how='left')

    # Add device usage ratios
    device_cols = ['is_mobile', 'is_tablet']
    for col in device_cols:
        if col in df.columns:
            device_ratio = grouped[col].mean().reset_index()
            device_ratio.columns = ['user_id', f'{col}_ratio']
            agg_df = agg_df.merge(device_ratio, on='user_id', how='left')

    # Add time pattern features
    time_features = ['hour', 'weekday', 'is_weekend']
    for col in time_features:
        if col in df.columns:
            time_mean = grouped[col].mean().reset_index()
            time_mean.columns = ['user_id', f'mean_{col}']
            agg_df = agg_df.merge(time_mean, on='user_id', how='left')

    # Add activity count
    activity_count = df.groupby('user_id').size().reset_index()
    activity_count.columns = ['user_id', 'activity_count']
    agg_df = agg_df.merge(activity_count, on='user_id', how='left')

    return agg_df

# Aggregate features for train and test
train_agg = aggregate_features(train_data)
test_agg = aggregate_features(test_data)

print("Feature aggregation completed!")
print("Train aggregated shape:", train_agg.shape)
print("Test aggregated shape:", test_agg.shape)

# Merge with labels
train = train_agg.merge(train_labels, on='user_id', how='inner')
print("After merging with labels - Train shape:", train.shape)

# Prepare test set (only users in test_users.csv)
test = test_agg[test_agg['user_id'].isin(test_users['user_id'])].copy()
print("Test set for prediction shape:", test.shape)

In [ ]:
print("\nStarting model training...")

# Prepare features and target
feature_cols = [col for col in train.columns if col not in ['user_id', 'target']]
X = train[feature_cols].copy()
y = train['target']
X_test = test[feature_cols].copy()

print("Feature columns:", len(feature_cols))
print("Target distribution:", y.value_counts())

# Handle categorical features
categorical_cols = [col for col in X.columns if col.startswith('mode_')]
print("Categorical columns to encode:", categorical_cols)

for col in categorical_cols:
    le = LabelEncoder()
    # Handle missing values and convert to string
    X[col] = X[col].fillna('Unknown').astype(str)
    X_test[col] = X_test[col].fillna('Unknown').astype(str)

    # Fit encoder on combined train+test data
    combined_values = pd.concat([X[col], X_test[col]], axis=0)
    le.fit(combined_values)

    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

# Handle missing values for all features
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Data preprocessing for modeling completed!")

# Train LightGBM model
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    metric='auc',
    verbose=-1
)

print("Training model...")
model.fit(X_imputed, y)

# Save model and preprocessing objects
joblib.dump(model, 'gender_classifier.pkl')
joblib.dump(imputer, 'imputer.pkl')
print("Model trained and saved successfully!")

# Model Evaluation
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# Split the training data for evaluation
X_train_eval, X_val_eval, y_train_eval, y_val_eval = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

# Train model on training subset
eval_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    metric='auc',
    verbose=-1
)

eval_model.fit(X_train_eval, y_train_eval)

# Make predictions on validation set
y_pred = eval_model.predict(X_val_eval)
y_pred_proba = eval_model.predict_proba(X_val_eval)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_val_eval, y_pred)
precision = precision_score(y_val_eval, y_pred)
recall = recall_score(y_val_eval, y_pred)
f1 = f1_score(y_val_eval, y_pred)

# Create metrics table
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Score': [accuracy, precision, recall, f1]
})

print("\nMODEL PERFORMANCE METRICS:")
print("-" * 30)
for _, row in metrics_df.iterrows():
    print(f"{row['Metric']:12}: {row['Score']:.4f}")

# Cross-validation scores
print("\nCROSS-VALIDATION RESULTS:")
print("-" * 30)
cv_scores = cross_val_score(eval_model, X_imputed, y, cv=5, scoring='accuracy')
print(f"CV Accuracy : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

cv_scores_f1 = cross_val_score(eval_model, X_imputed, y, cv=5, scoring='f1')
print(f"CV F1-Score : {cv_scores_f1.mean():.4f} (+/- {cv_scores_f1.std() * 2:.4f})")

# Confusion Matrix
print("\nCONFUSION MATRIX:")
print("-" * 20)
cm = confusion_matrix(y_val_eval, y_pred)
print(f"True Negatives : {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives : {cm[1,1]}")

# Detailed classification report
print("\nDETAILED CLASSIFICATION REPORT:")
print("-" * 40)
print(classification_report(y_val_eval, y_pred, target_names=['Female (0)', 'Male (1)']))

print("="*60)

# Generate Predictions
print("\nGenerating predictions...")
test_preds = model.predict_proba(X_test_imputed)[:, 1]

# Create submission file
submission = pd.DataFrame({
    'user_id': test['user_id'].values,
    'target': (test_preds > 0.5).astype(int)
})

In [ ]:
# Save predictions
submission.to_csv('submission.csv', index=False)
print(f"Submission file created with {len(submission)} predictions")
print("Target distribution in predictions:", submission['target'].value_counts())

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 most important features:")
print(feature_importance.head(10))

In [ ]:
# Generate Visualizations for Results
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create results directory if it doesn't exist
os.makedirs('results', exist_ok=True)

# 1. Feature Importance Bar Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=feature_importance.head(10))
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.savefig('results/feature_importance.png', dpi=300)
print("Saved: feature_importance.png")

# 2. Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Female', 'Male'], yticklabels=['Female', 'Male'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=300)
print("Saved: confusion_matrix.png")

# 3. Benchmark Table (improved styling)
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.axis('off')

# Create table
table = ax.table(cellText=metrics_df.values,
                 colLabels=metrics_df.columns,
                 cellLoc='center',
                 loc='center',
                 bbox=[0, 0, 1, 1])

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(10)

# Set column widths automatically
table.auto_set_column_width(col=list(range(len(metrics_df.columns))))

# Add borders to all cells and style header
for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(1)
    cell.set_edgecolor('black')
    if row == 0:  # header row
        cell.set_facecolor('#f0f0f0')
        cell.set_text_props(weight='bold')

plt.savefig('results/benchmark_comparison.png', dpi=300, bbox_inches='tight')
print("Saved: benchmark_comparison.png")

print("\nAll visualizations saved in the 'results/' folder.")

# (Optional) Download files automatically if in Google Colab
try:
    from google.colab import files
    import os
    print("\nDetected Colab environment. Downloading files...")
    for img in ['feature_importance.png', 'confusion_matrix.png', 'benchmark_comparison.png']:
        files.download(f'results/{img}')
    print("Download complete!")
except ImportError:
    print("\nNot running in Google Colab. Files are saved locally; you can manually copy them.")

In [ ]:
# Save Results
print("\nSaving results...")
!zip solution.zip gender_classifier.pkl imputer.pkl submission.csv
from google.colab import files
files.download('solution.zip')

print("Solution complete! Files saved and ready for download.")